# 🔬 برازش همه توزیع‌ها در Climatology Engine

این نوت‌بوک نحوه برازش همه توزیع‌های موجود را نشان می‌دهد.

**مواردی که یاد می‌گیرید:**
- بارگذاری پلاگین‌های توزیع
- برازش همه توزیع‌ها روی داده نمونه
- مقایسه نتایج با استفاده از AICc
- انتخاب بهترین مدل آماری
- رسم نمودار مقایسه AICc

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بارگذاری پلاگین‌های توزیع
plugins = load_plugins()
print(f'✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

# ذخیره در دیکشنری برای دسترسی آسان
distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# انتخاب داده tmean برای یک سال (۳۶۵ روز)
data_year = data[:365, 1]

print(f'📊 تعداد داده‌ها: {len(data_year)}')
print(f'   میانگین: {np.mean(data_year):.2f}°C')
print(f'   انحراف معیار: {np.std(data_year):.2f}°C')
print(f'   کمینه: {np.min(data_year):.2f}°C')
print(f'   بیشینه: {np.max(data_year):.2f}°C')

In [ ]:
# برازش همه توزیع‌ها
results_all = {}
print("\n🔄 در حال برازش توزیع‌ها...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        results_all[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}, "
              f"BIC = {res.get('bic', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")
        results_all[name] = None

print("\n✅ برازش همه توزیع‌ها کامل شد.")

In [ ]:
# انتخاب بهترین مدل (کمترین AICc)
valid_results = {k: v for k, v in results_all.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

if valid_results:
    best_name = min(valid_results, key=lambda x: valid_results[x]['aicc'])
    best_result = valid_results[best_name]

    print("=" * 60)
    print(f"🏆 بهترین توزیع: {best_name}")
    print(f"   AICc: {best_result['aicc']:.4f}")
    print(f"   BIC: {best_result.get('bic', np.nan):.4f}")
    print(f"   Log-likelihood: {best_result.get('loglik', np.nan):.4f}")
    print("=" * 60)
else:
    print("❌ هیچ توزیع معتبری یافت نشد.")

In [ ]:
# ایجاد جدول مقایسه
if valid_results:
    comparison_df = pd.DataFrame([{
        'توزیع': name,
        'AICc': res['aicc'],
        'BIC': res.get('bic', np.nan),
        'LogLik': res.get('loglik', np.nan),
        'ΔAICc': res['aicc'] - best_result['aicc'],
        'تعداد پارامتر': res.get('n_params', np.nan)
    } for name, res in valid_results.items()])

    comparison_df = comparison_df.sort_values('AICc').reset_index(drop=True)
    comparison_df.index = comparison_df.index + 1
    comparison_df
else:
    print("❌ داده‌ای برای نمایش وجود ندارد.")

In [ ]:
# رسم نمودار مقایسه AICc
if valid_results:
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#2ecc71' if name == best_name else '#e74c3c' 
              for name in comparison_df['توزیع']]
    bars = ax.barh(comparison_df['توزیع'], comparison_df['AICc'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.axvline(best_result['aicc'], color='black', linestyle='--', 
               linewidth=2, alpha=0.7, label=f'بهترین: {best_name}')
    
    ax.set_xlabel('AICc', fontsize=12)
    ax.set_title('مقایسه AICc توزیع‌های مختلف', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

    for i, (name, aicc) in enumerate(zip(comparison_df['توزیع'], comparison_df['AICc'])):
        ax.text(aicc + 0.5, i, f'{aicc:.1f}', va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("❌ داده‌ای برای رسم وجود ندارد.")

In [ ]:
# نمایش پارامترهای بهترین توزیع
if valid_results:
    print(f"📊 پارامترهای برازش برای توزیع {best_name}:")
    print("=" * 50)
    for key, value in best_result.items():
        if isinstance(value, float):
            print(f"   {key}: {value:.6f}")
        else:
            print(f"   {key}: {value}")
    print("=" * 50)
else:
    print("❌ داده‌ای برای نمایش وجود ندارد.")

In [ ]:
# رسم منحنی‌های توزیع همه مدل‌ها
if valid_results:
    fig, ax = plt.subplots(figsize=(12, 7))

    # هیستوگرام داده
    ax.hist(data_year, bins=30, density=True, alpha=0.3, 
            color='gray', edgecolor='black', label='داده')

    # رنگ‌های مختلف برای توزیع‌ها
    colors_list = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    x = np.linspace(min(data_year), max(data_year), 500)

    for i, (name, res) in enumerate(valid_results.items()):
        try:
            # محاسبه PDF برای هر توزیع (با استفاده از تابع pdf موجود در پلاگین)
            dist = distributions[name]
            if hasattr(dist, 'pdf'):
                params = {p: res[p] for p in dist.params if p in res}
                pdf_vals = dist.pdf(x, params)
                ax.plot(x, pdf_vals, color=colors_list[i % len(colors_list)], 
                        linewidth=2, label=f'{name} (AICc={res["aicc"]:.1f})')
            else:
                print(f"⚠️ توزیع {name} متد pdf ندارد.")
        except Exception as e:
            print(f"⚠️ خطا در رسم {name}: {str(e)}")

    ax.set_xlabel('دما (°C)', fontsize=12)
    ax.set_ylabel('چگالی احتمال', fontsize=12)
    ax.set_title('مقایسه منحنی‌های توزیع برازش شده', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("❌ داده‌ای برای رسم وجود ندارد.")

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ بارگذاری پلاگین‌های توزیع با `load_plugins()`
✅ برازش همه توزیع‌های موجود روی داده نمونه
✅ مقایسه نتایج با استفاده از معیار AICc
✅ انتخاب بهترین مدل آماری
✅ رسم نمودار مقایسه AICc
✅ رسم منحنی‌های توزیع برازش شده

---

**مراحل بعدی:**
- نوت‌بوک ۰۴: انتخاب مدل (AIC, BIC, AICc)
- نوت‌بوک ۰۵: کنترل کیفیت (Quality Flag)
- نوت‌بوک ۰۶: عدم‌قطعیت Bootstrap

---

**نکته مهم:**
معیار AICc برای مقایسه مدل‌هایی با تعداد پارامترهای مختلف مناسب است. هرچه AICc کمتر باشد، مدل بهتر است.